In [4]:
import yaml
from pathlib import Path

def load_dag(file_name: Path) -> dict:
    """Load YAML file."""
    with open(file_name) as f: 
        return yaml.safe_load(f)

def load_dags(dir_name: Path) -> dict:
    """Load YAML files."""
    dags = {}

    for f in dir_name.rglob("*.yml"):
        dags[str(f)] = load_dag(f)
        
    for f in dir_name.rglob("*.yaml"):
        dags[str(f)] = load_dag(f)

    return dags

def extract_tasks(dag_list: list) -> list:
    """Extract tasks from DAG definition."""
    
    tasks = []

    for dag_file, dag_obj in dag_list.items():

        dag_name = list(dag_obj.keys())[0]
        dag_def = dag_obj[dag_name]
        
        for task in dag_def.get('tasks', None):
            
            id = task.get('id', None)
            operator = task.get('op_type', None)
            file = task.get('source_file', None)
            params = task.get('params', None)

            tasks.append((dag_name, id, operator, file, params))

    return tasks

In [ ]:
import jinja2
from pathlib import Path

def load_sql(sql_file: Path) -> str:
    """Load and return SQL test from a file path."""
    with open(sql_file, "r", encoding="utf-8") as f:
        return f.read()
    
def load_sqls(dir_name: Path) -> dict:
    """Load SQL files."""
    sqls = {}

    for f in dir_name.rglob("*.sql"):
        sqls[str(f)] = load_sql(f)

    return sqls
    
def format_sql(sql: str, context: dict) -> str:
    """Format SQL using jinja templating if necessary"""
    return jinja2.Environment().from_string(sql).render(context)

In [75]:
from sqlglot import parse_one, exp

In [97]:
def_path = Path('/Users/adamlightner/Documents/Projects/data-warehouse-analysis-tool/examples/definitions')
sql_path = Path('/Users/adamlightner/Documents/Projects/data-warehouse-analysis-tool/examples/sql')

In [98]:
yamls = load_dags(def_path)
sqls = load_sqls(sql_path)

In [99]:
yamls

{'/Users/adamlightner/Documents/Projects/data-warehouse-analysis-tool/examples/definitions/teams.yml': {'load_teams': {'catchup': False,
   'owner': 'airflow',
   'schedule': '0 0 1 9 *',
   'max_active_runs': 1,
   'max_retries': 2,
   'tasks': [{'id': 'get_batch',
     'op_type': 'PythonOperator',
     'source_file': 'utils/get_batch.py'},
    {'id': 'update_dim_table',
     'op_type': 'SnowflakeOperator',
     'pool': 'nfl',
     'source_file': 'examples/sql/dimension/dim_team.sql',
     'depends_on': ['get_batch'],
     'params': {'TARGET_TABLE': 'DIMENSION.NFL_TEAM',
      'SOURCE_TABLE': 'INGESTION.NFL_TEAM'}}]}},
 '/Users/adamlightner/Documents/Projects/data-warehouse-analysis-tool/examples/definitions/games.yml': {'load_games': {'catchup': False,
   'owner': 'airflow',
   'schedule': '0 0 1 9 *',
   'max_active_runs': 1,
   'max_retries': 2,
   'tasks': [{'id': 'get_batch',
     'op_type': 'PythonOperator',
     'source_file': 'utils/get_batch.py'},
    {'id': 'insert_to_staing

In [100]:
sqls

{'/Users/adamlightner/Documents/Projects/data-warehouse-analysis-tool/examples/sql/staging/stg_game.sql': 'insert into {{ TARGET_TABLE }} (\n    game_id,\n    game_date,\n    home_team_id,\n    home_score,\n    vis_team_id,\n    vis_score\n)\nselect\n    upper(src.id) as game_id,\n    timestamp(src.game_time) as game_date,\n    upper(src.home_team) as home_team_id,\n    src.home_score,\n    upper(src.vis_team) as vis_team_id,\n    src.vis_score\nfrom\n    {{ SOURCE_TABLE }} src\n;\n',
 '/Users/adamlightner/Documents/Projects/data-warehouse-analysis-tool/examples/sql/fact/fct_game.sql': 'insert into {{ TARGET_TABLE }} (\n    game_id,\n    game_date,\n    home_team_id,\n    home_team_name,\n    home_score,\n    vis_team_id,\n    vis_team_name,\n    vis_score\n)\nselect\n    gm.game_id,\n    gm.game_date,\n    gm.home_team_id,\n    home_t.team_name as home_team_name,\n    gm.home_score,\n    gm.vis_team_id,\n    vis_t.team_name as vis_team_name,\n    gm.vis_score\nfrom\n    {{ SOURCE_TABL

In [101]:
tasks = []

for yml, dag in yamls.items(): 

    print(yml)
    # print(dag)

    for name, definition in dag.items():
        print(name)
        # print(definition)

        for task in definition['tasks']:

            print(task)
            tasks.append(task)

/Users/adamlightner/Documents/Projects/data-warehouse-analysis-tool/examples/definitions/teams.yml
load_teams
{'id': 'get_batch', 'op_type': 'PythonOperator', 'source_file': 'utils/get_batch.py'}
{'id': 'update_dim_table', 'op_type': 'SnowflakeOperator', 'pool': 'nfl', 'source_file': 'examples/sql/dimension/dim_team.sql', 'depends_on': ['get_batch'], 'params': {'TARGET_TABLE': 'DIMENSION.NFL_TEAM', 'SOURCE_TABLE': 'INGESTION.NFL_TEAM'}}
/Users/adamlightner/Documents/Projects/data-warehouse-analysis-tool/examples/definitions/games.yml
load_games
{'id': 'get_batch', 'op_type': 'PythonOperator', 'source_file': 'utils/get_batch.py'}
{'id': 'insert_to_staing', 'op_type': 'SnowflakeOperator', 'pool': 'nfl', 'source_file': 'examples/sql/staging/stg_game.sql', 'params': {'TARGET_TABLE': 'STAGING.NFL_GAME', 'SOURCE_TABLE': 'INGESTION.NFL_GAME'}, 'depends_on': ['get_batch']}
{'id': 'insert_to_fact', 'op_type': 'SnowflakeOperator', 'pool': 'nfl', 'source_file': 'examples/sql/fact/fct_game.sql', '

In [102]:
sqls

{'/Users/adamlightner/Documents/Projects/data-warehouse-analysis-tool/examples/sql/staging/stg_game.sql': 'insert into {{ TARGET_TABLE }} (\n    game_id,\n    game_date,\n    home_team_id,\n    home_score,\n    vis_team_id,\n    vis_score\n)\nselect\n    upper(src.id) as game_id,\n    timestamp(src.game_time) as game_date,\n    upper(src.home_team) as home_team_id,\n    src.home_score,\n    upper(src.vis_team) as vis_team_id,\n    src.vis_score\nfrom\n    {{ SOURCE_TABLE }} src\n;\n',
 '/Users/adamlightner/Documents/Projects/data-warehouse-analysis-tool/examples/sql/fact/fct_game.sql': 'insert into {{ TARGET_TABLE }} (\n    game_id,\n    game_date,\n    home_team_id,\n    home_team_name,\n    home_score,\n    vis_team_id,\n    vis_team_name,\n    vis_score\n)\nselect\n    gm.game_id,\n    gm.game_date,\n    gm.home_team_id,\n    home_t.team_name as home_team_name,\n    gm.home_score,\n    gm.vis_team_id,\n    vis_t.team_name as vis_team_name,\n    gm.vis_score\nfrom\n    {{ SOURCE_TABL

In [103]:
sql_tasks = [(f"/Users/adamlightner/Documents/Projects/data-warehouse-analysis-tool/{t['source_file']}", t['params']) for t in tasks if t['op_type'] == 'SnowflakeOperator']

In [104]:
sql_tasks

[('/Users/adamlightner/Documents/Projects/data-warehouse-analysis-tool/examples/sql/dimension/dim_team.sql',
  {'TARGET_TABLE': 'DIMENSION.NFL_TEAM',
   'SOURCE_TABLE': 'INGESTION.NFL_TEAM'}),
 ('/Users/adamlightner/Documents/Projects/data-warehouse-analysis-tool/examples/sql/staging/stg_game.sql',
  {'TARGET_TABLE': 'STAGING.NFL_GAME', 'SOURCE_TABLE': 'INGESTION.NFL_GAME'}),
 ('/Users/adamlightner/Documents/Projects/data-warehouse-analysis-tool/examples/sql/fact/fct_game.sql',
  {'TARGET_TABLE': 'FACT.NFL_GAME',
   'SOURCE_TABLE': 'STAGING.NFL_GAME',
   'TEAM_TABLE': 'STAGING.NFL_GAME'})]

In [105]:
for st in sql_tasks:

    # print(st)
    
    sqls[st[0]] = format_sql(sqls[st[0]], st[1])

In [106]:
sqls

{'/Users/adamlightner/Documents/Projects/data-warehouse-analysis-tool/examples/sql/staging/stg_game.sql': 'insert into STAGING.NFL_GAME (\n    game_id,\n    game_date,\n    home_team_id,\n    home_score,\n    vis_team_id,\n    vis_score\n)\nselect\n    upper(src.id) as game_id,\n    timestamp(src.game_time) as game_date,\n    upper(src.home_team) as home_team_id,\n    src.home_score,\n    upper(src.vis_team) as vis_team_id,\n    src.vis_score\nfrom\n    INGESTION.NFL_GAME src\n;',
 '/Users/adamlightner/Documents/Projects/data-warehouse-analysis-tool/examples/sql/fact/fct_game.sql': 'insert into FACT.NFL_GAME (\n    game_id,\n    game_date,\n    home_team_id,\n    home_team_name,\n    home_score,\n    vis_team_id,\n    vis_team_name,\n    vis_score\n)\nselect\n    gm.game_id,\n    gm.game_date,\n    gm.home_team_id,\n    home_t.team_name as home_team_name,\n    gm.home_score,\n    gm.vis_team_id,\n    vis_t.team_name as vis_team_name,\n    gm.vis_score\nfrom\n    STAGING.NFL_GAME gm\nle

In [107]:
# dir(exp)

In [213]:
def get_transaction_type(sql: str) -> str:

    tree = parse_one(sql)

    print(type(tree))

    if isinstance(tree, exp.Insert):
        return sql_insert(tree)
    if isinstance(tree, exp.Merge):
        return None
    if isinstance(tree, exp.Update):
        return None
    if isinstance(tree, exp.Copy):
        return None

In [214]:
def sql_insert(tree):

    res = {}

    target = tree.this.this.sql()
    res['target'] = target

    columns_inserted = [c.this for c in tree.this.expressions]
    res['columns_inserted'] = columns_inserted

    table_definition = tree.expression.sql()
    res['table_definition'] = table_definition

    table_deps = set([f"{table.db}.{table.this.sql()}" for table in tree.expression.find_all(exp.Table)])
    res['table_deps'] = table_deps

    return res


In [217]:
objs = []

for k, v in sqls.items():

    print(k)
    print(v)

    # objs.append(parse_one(v))
    
    sql_parts = get_transaction_type(v)
    print(sql_parts)


    print('-'*100)

/Users/adamlightner/Documents/Projects/data-warehouse-analysis-tool/examples/sql/staging/stg_game.sql
insert into STAGING.NFL_GAME (
    game_id,
    game_date,
    home_team_id,
    home_score,
    vis_team_id,
    vis_score
)
select
    upper(src.id) as game_id,
    timestamp(src.game_time) as game_date,
    upper(src.home_team) as home_team_id,
    src.home_score,
    upper(src.vis_team) as vis_team_id,
    src.vis_score
from
    INGESTION.NFL_GAME src
;
<class 'sqlglot.expressions.Insert'>
{'target': 'STAGING.NFL_GAME', 'columns_inserted': ['game_id', 'game_date', 'home_team_id', 'home_score', 'vis_team_id', 'vis_score'], 'table_definition': 'SELECT UPPER(src.id) AS game_id, TIMESTAMP(src.game_time) AS game_date, UPPER(src.home_team) AS home_team_id, src.home_score, UPPER(src.vis_team) AS vis_team_id, src.vis_score FROM INGESTION.NFL_GAME AS src', 'table_deps': {'INGESTION.NFL_GAME'}}
--------------------------------------------------------------------------------------------------

In [219]:
# objs[1]